In [1]:
import os

In [2]:
%pwd

'c:\\Users\\HP\\Desktop\\Python\\Vs_Python\\MLoPs\\Movie_recommendation_system\\Notebook'

In [3]:
os.chdir("../")

In [5]:
%pwd

'c:\\Users\\HP\\Desktop\\Python\\Vs_Python\\MLoPs\\Movie_recommendation_system'

In [ ]:
from dataclasses import dataclass
from pathlib import Path

@dataclass(frozen=True)
class ModelEvaluationConfig:
    root_dir: Path
    transformed_data_path: Path
    similarity_path: Path

In [7]:
from Movie_Recommendation_system.constants import *
from Movie_Recommendation_system.utils.common import read_yaml, create_directories

In [8]:
class ConfigurationManager:
    def __init__(
        self,
        config_filepath = CONFIG_FILE_PATH,
        params_filepath = PARAMS_FILE_PATH,
        schema_filepath = SCHEMA_FILE_PATH):

        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)
        self.schema = read_yaml(schema_filepath)

        create_directories([self.config.artifacts_root])

    def get_model_evaluation_config(self) -> ModelEvaluationConfig:
        config = self.config.model_evaluation

        create_directories([Path(config.root_dir)])

        return ModelEvaluationConfig(
            root_dir=Path(config.root_dir),
            transformed_data_path=Path(config.transformed_data_path),
            similarity_path=Path(config.similarity_path),
        )

In [13]:
import pandas as pd
import pickle
from pathlib import Path

class ModelEvaluation:
    def __init__(self, config: ModelEvaluationConfig):
        self.config = config

    def evaluate_model(self):
        movies = pd.read_csv(self.config.transformed_data_path)

        with open(self.config.similarity_path, "rb") as f:
            similarity = pickle.load(f)

        test_movies = ["Avatar", "Batman Begins", "Titanic"]

        print("\n🔎 Model Evaluation Results:\n")

        for movie_name in test_movies:
            if movie_name not in movies["title"].values:
                print(f"{movie_name} not found")
                continue

            index = movies[movies["title"] == movie_name].index[0]
            distances = similarity[index]

            movie_list = sorted(
                list(enumerate(distances)),
                reverse=True,
                key=lambda x: x[1]
            )[1:6]

            print(f"\nTop recommendations for {movie_name}:")

            for i in movie_list:
                print("  -", movies.iloc[i[0]].title)

In [14]:
try:
    config = ConfigurationManager()
    model_eval_config = config.get_model_evaluation_config()

    model_evaluation = ModelEvaluation(model_eval_config)
    model_evaluation.evaluate_model()

except Exception as e:
    raise e

[2026-02-22 11:17:23,108: INFO: common: yaml file: config\config.yaml loaded successfully]
[2026-02-22 11:17:23,114: INFO: common: yaml file: params.yaml loaded successfully]
[2026-02-22 11:17:23,125: INFO: common: yaml file: schema.yaml loaded successfully]
[2026-02-22 11:17:23,127: INFO: common: created directory at: artifacts]
[2026-02-22 11:17:23,131: INFO: common: created directory at: artifacts\model_evaluation]

🔎 Model Evaluation Results:


Top recommendations for Avatar:
  - Aliens
  - Moonraker
  - Alien
  - Alien³
  - Silent Running

Top recommendations for Batman Begins:
  - The Dark Knight
  - The Dark Knight Rises
  - Batman
  - Batman & Robin
  - Batman Returns

Top recommendations for Titanic:
  - The Notebook
  - Romance & Cigarettes
  - Captain Phillips
  - Veer-Zaara
  - Four Weddings and a Funeral
